# J-Space Toolkit Demo

This notebook trains a Jacobian Lens on a small decoder-only model and replicates a paper sanity check.

In [ ]:
import matplotlib.pyplot as plt
import torch

from jspace.discovery import compute_discovery_metrics, infer_workspace_boundaries
from jspace.jacobian_lens import train_jacobian_lens
from jspace.model_adapter import get_unembedding_matrix, layer_indices, load_model, normalize_fn
from jspace.readout import lens_readout


In [ ]:
MODEL_NAME = "sshleifer/tiny-gpt2"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, tokenizer = load_model(MODEL_NAME, device, torch.float32)
layers = layer_indices(model)
print(f"Loaded {MODEL_NAME} with {len(layers)} layers")

In [ ]:
# Prepare a tiny corpus
prompts = [
    "The cat sat on the mat.",
    "In 1950, scientists discovered that robots can calculate.",
    "What color is the sky? The answer is",
    "Once upon a time, a curious robot decided to",
] * 4
enc = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=32)
corpus_ids = enc["input_ids"]

In [ ]:
# Train the J-Lens
J = train_jacobian_lens(
    model,
    corpus_ids,
    target_layer=layers[-2],
    dtype=torch.float32,
    max_positions=32,
    batch_size=2,
    output_dim_chunk=16,
)
print(f"Trained J_l for layers {list(J.keys())}")

## Verbal Report Sanity Check

"Think of a sport:" - the J-Lens at the colon position should surface the sport the model outputs.

In [ ]:
prompt = "Think of a sport:"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

with torch.no_grad():
    generated = model.generate(input_ids, max_new_tokens=5, do_sample=False)
answer = tokenizer.decode(generated[0, input_ids.shape[1]:], skip_special_tokens=True).strip().split()[0]
print(f"Model answer: {answer}")

outputs = model(input_ids, output_hidden_states=True, return_dict=True)
layer = layers[len(layers)//2]
h = outputs.hidden_states[layer + 1][0, -1]
W_U = get_unembedding_matrix(model)
topk_indices, topk_probs = lens_readout(h, J[layer], W_U, normalize_fn(model), tokenizer, top_k=10)
topk_tokens = [tokenizer.decode([idx]) for idx in topk_indices.tolist()]
print("J-Lens top-10 at colon position:")
for tok, p in zip(topk_tokens, topk_probs.tolist(), strict=True):
    print(f"  {tok!r}: {p:.4f}")

## Workspace Layer Auto-Discovery

In [ ]:
metrics = compute_discovery_metrics(
    model, tokenizer, J, corpus_ids, W_U, normalize_fn(model)
)
start, end = infer_workspace_boundaries(metrics)
print(f"Inferred workspace band: [{start}, {end}]")

In [ ]:
# Plot CKA block structure
plt.figure(figsize=(6, 5))
plt.imshow(metrics["cka_block"], cmap="viridis", vmin=0, vmax=1)
plt.colorbar(label="CKA similarity")
plt.xlabel("Layer")
plt.ylabel("Layer")
plt.title("CKA similarity of J-lens vectors by layer")
plt.show()

In [ ]:
# Plot per-layer metrics
fig, axes = plt.subplots(1, 3, figsize=(15, 3))
axes[0].plot(metrics["kurtosis"])
axes[0].set_title("Excess kurtosis")
axes[0].set_xlabel("Layer")
axes[1].plot(metrics["accuracy"])
axes[1].set_title("Next-token accuracy proxy")
axes[1].set_xlabel("Layer")
axes[2].plot(metrics["autocorr"])
axes[2].set_title("Top-1 token autocorrelation")
axes[2].set_xlabel("Layer")
plt.tight_layout()
plt.show()